In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
        .appName("Final_combined_challenge")\
        .getOrCreate()
print("Spark Session created")

Spark Session created


In [2]:
from google.colab import files
uploaded=files.upload()

Saving employees.csv to employees.csv
Saving logins.txt to logins.txt
Saving orders.json to orders.json
Saving sales.csv to sales.csv


In [3]:
!ls

employees.csv  logins.txt  orders.json	sales.csv  sample_data


In [ ]:
from pyspark.sql.functions import explode, col
df_sales = spark.read.csv("sales.csv", header=True, inferSchema=True)
df_emp = spark.read.csv("employees.csv", header=True, inferSchema=True)
df= spark.read.text("logins.txt")
df_names=df.withColumnRenamed("value", "Name")

df_o= spark.read.option("multiline", "true").json("orders.json")

df_orders= df_o.select(explode(col("orders")).alias("order")).select("order.*")
df_ord=df_orders.select("order_id", "customer", "product", "amount", "city")

df_ord.show()
df_names.show()
df_emp.show()
df_sales.show()

In [ ]:
#Question 1
df_emp.join(df_sales, on="emp_id", how="inner").show()

In [ ]:
#Question 2
from pyspark.sql.functions import col ,min, max, avg, sum

df_emp.join(df_sales, on="emp_id", how="inner")\
.withColumn("revenue", col("quantity")*col("price"))\
.groupby("emp_id","name")\
.agg(sum("revenue").alias("Total_rev")).show()


In [ ]:
#Question 3
df_emp.join(df_sales, on="emp_id", how="inner")\
      .withColumn("revenue", col("quantity") * col("price"))\
      .groupby("emp_id","name")\
      .agg(sum("revenue").alias("Total_sales"))\
      .orderBy(col("Total_sales").desc()).show(5)

In [ ]:
#Question 4
df_emp.join(df_sales, on="emp_id", how="inner")\
      .withColumn("revenue", col("quantity") * col("price"))\
      .groupby("department")\
      .agg(sum("revenue").alias("Total_rev"))\
      .orderBy(col("Total_rev").desc()).show(1)

In [ ]:
#Question 5
df_result= df_emp.join(df_sales, on="emp_id", how="inner")\
      .withColumn("revenue", col("quantity") * col("price"))\
      .groupby("name")\
      .agg(sum("revenue").alias("Total_rev"))\
      .orderBy(col("Total_rev").desc())

df_result.show()

df_result.coalesce(1).write.mode("overwrite").option("header",True).csv("final_sales_report")